In [1]:
import polars as pl
import glob
import os.path

In [2]:
SCALED=1000
HASH_THRESHOLD=int(20000 / SCALED)


#CSVPATH=f'../outputs.mapping/cds/singlehash.k21/outputs.3216.min{MIN}/manysearch.3216.csv'
CSVPATH=f'/home/ctbrown/scratch3/2025-workflow-core99/outputs.cds/cds3/manysearch.cds3.3216.csv'

In [3]:
dflist = []
#ms_filenames = glob.glob('../manysearch-3216/*.cds.x.3216.manysearch.csv')
ms_filenames = glob.glob(CSVPATH)
for filename in ms_filenames:
    print(os.path.basename(filename))
    df = pl.read_csv(filename)
    dflist.append(df)

df = pl.concat(dflist)
df

manysearch.cds3.3216.csv


query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""s__Lactobacillus amylovorus""","""126daabd3e4858fd7f57ebee5c07f5…","""SRR11125501""",0.041928,167,21,1000,"""DNA""","""16d0ec735d0ee9329df5986e7d5fef…",0.00081,0.041928,9.766467,10.0,5.717864,0.859815,0.713125,0.78647,0.859815,1631,914978
"""s__JALFVM01 sp022787145""","""6e456de504903038c251145eeed987…","""SRR11125501""",0.001421,4,21,1000,"""DNA""","""16d0ec735d0ee9329df5986e7d5fef…",0.000019,0.001421,1.25,1.0,0.433013,0.731827,0.597023,0.664425,0.731827,5,914978
"""s__Bariatricus sp004560705""","""a053514b987f8c11e2ed518938e366…","""SRR11125501""",0.000989,3,21,1000,"""DNA""","""16d0ec735d0ee9329df5986e7d5fef…",0.000015,0.000989,1.333333,1.0,0.471405,0.7193,0.5889,0.6541,0.7193,4,914978
"""s__Fimisoma sp002320005""","""019f828d64f7de39e3cf407b3f3d15…","""SRR11125501""",0.059783,110,21,1000,"""DNA""","""16d0ec735d0ee9329df5986e7d5fef…",0.000539,0.059783,1.327273,1.0,0.619517,0.874463,0.699087,0.786775,0.874463,146,914978
"""s__Holdemanella porci""","""398766a16d18d486ff57f019d264e4…","""SRR11125501""",0.037491,52,21,1000,"""DNA""","""16d0ec735d0ee9329df5986e7d5fef…",0.000255,0.037491,1.634615,1.0,0.920469,0.855247,0.674584,0.764916,0.855247,85,914978
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""s__Lactobacillus amylovorus""","""126daabd3e4858fd7f57ebee5c07f5…","""SRR17241654""",0.074065,295,21,1000,"""DNA""","""97e014afa768774ea86ad500ff6d50…",0.000068,0.074065,14.735593,7.0,30.834571,0.88343,0.633314,0.758372,0.88343,4347,32465693
"""s__Fimisoma sp002320005""","""019f828d64f7de39e3cf407b3f3d15…","""SRR17241654""",0.529891,975,21,1000,"""DNA""","""97e014afa768774ea86ad500ff6d50…",0.000225,0.529891,9.934359,9.0,6.207506,0.970211,0.670413,0.820312,0.970211,9686,32465693
"""s__Sodaliphilus sp004557565""","""01b84a101775aec06b80aef75f7a9b…","""SRR17241654""",0.169446,1829,21,1000,"""DNA""","""97e014afa768774ea86ad500ff6d50…",0.000422,0.169446,9.295243,5.0,38.515954,0.91894,0.6908,0.80487,0.91894,17001,32465693


In [7]:
df['query_name'].unique()

query_name
str
"""s__Roseburia inulinivorans"""
"""s__Lactobacillus amylovorus"""
"""s__JAFBIX01 sp021531895"""
"""s__JALFVM01 sp022787145"""
"""s__Fimisoma sp002320005"""
…
"""s__Phascolarctobacterium_A suc…"
"""s__Mogibacterium_A kristiansen…"
"""s__Cryptobacteroides sp9005469…"


## Summarize by percent across samples.

In [4]:
for row in df.filter(pl.col("intersect_hashes") >= HASH_THRESHOLD)['query_name'].value_counts().with_columns(
    (pl.col("count") / 3216 * 100).alias("percent")
).sort(by='percent', descending=True).iter_rows(named=True):
    print(f"{row['percent']:.1f}% {row['query_name']} ")

99.0% s__Sodaliphilus sp004557565 
98.5% s__Lactobacillus amylovorus 
97.5% s__Cryptobacteroides sp900546925 
97.1% s__Fimisoma sp002320005 
97.0% s__Mogibacterium_A kristiansenii 
96.7% s__UBA2868 sp004552595 
96.4% s__Prevotella sp002251295 
96.3% s__JAFBIX01 sp021531895 
95.7% s__Bariatricus sp004560705 
93.8% s__Prevotella sp000434975 
90.6% s__JALFVM01 sp022787145 
90.1% s__Holdemanella porci 
87.6% s__Phascolarctobacterium_A succinatutens 
82.7% s__Gemmiger qucibialis 
77.1% s__Roseburia inulinivorans 


In [5]:
SUB_SPECIES2=(
             's__Cryptobacteroides sp900546925',
             's__Phascolarctobacterium_A succinatutens',
             's__Mogibacterium_A kristiansenii',
             's__Prevotella sp002251295',
#             's__Prevotella sp000434975',
#             's__Holdemanella porci',
)

for row in df.filter(pl.col("intersect_hashes") >= HASH_THRESHOLD)['query_name'].value_counts().with_columns(
    (pl.col("count") / 3216 * 100).alias("percent")
).sort(by='percent').iter_rows(named=True):
    if row["query_name"] in SUB_SPECIES2:
        print(f"{row['percent']:.1f}% {row['query_name']} ")

87.6% s__Phascolarctobacterium_A succinatutens 
96.4% s__Prevotella sp002251295 
97.0% s__Mogibacterium_A kristiansenii 
97.5% s__Cryptobacteroides sp900546925 


In [6]:
for row in df.filter(pl.col("intersect_hashes") >= HASH_THRESHOLD)['query_name'].value_counts().with_columns(
    (pl.col("count") / 3216 * 100).alias("percent")
).sort(by='percent').iter_rows(named=True):
    print(f"{row['percent']:.1f}% {row['query_name']} ")

77.1% s__Roseburia inulinivorans 
82.7% s__Gemmiger qucibialis 
87.6% s__Phascolarctobacterium_A succinatutens 
90.1% s__Holdemanella porci 
90.6% s__JALFVM01 sp022787145 
93.8% s__Prevotella sp000434975 
95.7% s__Bariatricus sp004560705 
96.3% s__JAFBIX01 sp021531895 
96.4% s__Prevotella sp002251295 
96.7% s__UBA2868 sp004552595 
97.0% s__Mogibacterium_A kristiansenii 
97.1% s__Fimisoma sp002320005 
97.5% s__Cryptobacteroides sp900546925 
98.5% s__Lactobacillus amylovorus 
99.0% s__Sodaliphilus sp004557565 
